# 02 — Fine-tuning do CodeBERT

Objetivo: entender o ciclo de treino na prática.

- O que acontece em cada epoch
- Por que a train_loss desce e quando val_loss diverge (overfitting)
- Como o MLflow rastreia experimentos
- O que o modelo erra (análise de erros)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('../src').resolve()))

import mlflow
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams['figure.dpi'] = 120

## 1. Iniciar treino

Execute a célula abaixo e observe:
- `train_loss` descer a cada epoch (o modelo está aprendendo)
- `val_loss` deve acompanhar — se divergir, é overfitting
- `checkpoint salvo` aparece quando val_loss melhora

In [ ]:
from train import train

train(
    num_epochs=5,
    batch_size=16,
    learning_rate=2e-5,
    max_length=256,
    patience=3,
)

## 2. Visualizar curvas de loss via MLflow

`mlflow ui` no terminal para abrir o dashboard. Aqui vamos ler os dados diretamente.

In [ ]:
import mlflow

mlflow.set_tracking_uri("sqlite:///" + str(Path('..').resolve() / 'mlflow.db'))
client = mlflow.tracking.MlflowClient()

exp = client.get_experiment_by_name('code-review-classifier')
runs = client.search_runs(experiment_ids=[exp.experiment_id], order_by=['start_time DESC'])
run = runs[0]

print(f'Run ID: {run.info.run_id}')
print(f'Params: {run.data.params}')
print(f'Final metrics: {run.data.metrics}')

In [ ]:
def get_metric_history(run_id, metric_name):
    history = client.get_metric_history(run_id, metric_name)
    return [(h.step, h.value) for h in sorted(history, key=lambda x: x.step)]

run_id = runs[0].info.run_id
train_loss = get_metric_history(run_id, 'train_loss')
val_loss   = get_metric_history(run_id, 'val_loss')
val_acc    = get_metric_history(run_id, 'val_accuracy')

epochs_tl  = [e for e, _ in train_loss]
epochs_vl  = [e for e, _ in val_loss]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs_tl, [v for _, v in train_loss], 'o-', label='train_loss', color='steelblue')
ax1.plot(epochs_vl, [v for _, v in val_loss],   'o-', label='val_loss',   color='coral')
ax1.set_title('Train vs Val Loss por Epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

ax2.plot([e for e, _ in val_acc], [v for _, v in val_acc], 'o-', color='mediumseagreen')
ax2.set_title('Val Accuracy por Epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_ylim(0, 1)
ax2.grid(True)

plt.tight_layout()
plt.show()

## 3. Avaliar no test set

In [ ]:
from evaluate import evaluate
from pathlib import Path

results = evaluate(checkpoint_path=Path('../models/full_finetuned'))
print(f"\nF1 Macro: {results['f1_macro']:.4f}")
print(f"F1 Weighted: {results['f1_weighted']:.4f}")

## 4. Inferência manual

Teste o modelo com findings que você escrever.

In [ ]:
import torch
from model import load_finetuned, LABELS

model, tokenizer = load_finetuned('../models/full_finetuned')
model.eval()

def predict(text: str):
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=256)
    with torch.no_grad():
        logits = model(**enc).logits
    probs = torch.softmax(logits, dim=-1).squeeze()
    pred_id = probs.argmax().item()
    return {
        'class': LABELS[pred_id],
        'confidence': round(probs[pred_id].item(), 4),
        'all_scores': {l: round(probs[i].item(), 4) for i, l in enumerate(LABELS)},
    }

test_findings = [
    "SQL query built with string concatenation — SQL injection risk",
    "This class violates single responsibility — extract domain logic",
    "No logging on exception path — impossible to diagnose in production",
    "Variable name 'x' is ambiguous — use more descriptive name",
    "This pattern is intentional — the retry handles transient failures by design",
]

for finding in test_findings:
    result = predict(finding)
    print(f"  [{result['class']:15s} {result['confidence']:.2%}] {finding[:70]}")